## Working with RAG 

### Basic Workflow of RAG -> 
1. First we need to load the all the folder/ subfolders into the documents object of langchain for this we use the langchain_community.documents_loader import DirectoryLoader, TextLoader 

In [13]:
import os
import glob 
import tiktoken
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from openai import OpenAI
import tiktoken
from langchain_community.document_loaders import  DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from sklearn.manifold import TSNE
import plotly.graph_objects as go
import glob
from google import genai


In [11]:
model_name = "gemini-2.5-flash"
db_name = "vector_db"

In [9]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path,recursive=True)

print(f"Total file found in knowledge : {len(files)}")

entire_knowledge_base = []
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"
    
print(f"Total character in entire knowledge base : {len(entire_knowledge_base)}")



Total file found in knowledge : 76
Total character in entire knowledge base : 304434


In [19]:
encoding = tiktoken.encoding_for_model("gpt-4.1-nano")
token = encoding.encode(str(entire_knowledge_base))
print(f"The total number of token in entire knowledge base : {len(token)}")


The total number of token in entire knowledge base : 840170


In [ ]:
client = genai.Client()
response = client.models.count_tokens(
    model = "model_name",
    contents = entire_knowledge_base

)
token_count = response.total_tokens
print(f"Total token for {model_name}: {token_count}")

### Below we have use  langchain_community.document_loaders import  DirectoryLoader, TextLoader

In [ ]:
folders = glob.glob("knowledge-base/*")




documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    folder_doc = loader.load()
    for doc in folder_doc:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap =200)
chunk = text_splitter.split_documents(documents)
print(f"The total chunks : {len(chunk)}")
print(chunk[0])
print(chunk[1])


The total chunks : 413
page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}
page_content='However, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 emplo